# Remote CMA-ES training — AutonomousCar

Runs the B1 gap-policy CMA-ES campaign on this machine's CPU cores.
MuJoCo rollouts are CPU-bound; the A40 GPU is *not used* by this pipeline
(physics + NumPy policy only) — the win here is the Xeon core count, which
the launcher auto-detects (one rollout worker per core).

Flow: **Setup → Launch → Monitor → Validate → Download results**.


In [ ]:
# 1) SETUP — run once per machine
import os
import sys

BRANCH = "vibe/cma-robust-speed-911692"   # training code + new genome
REPO_URL = "https://github.com/merwynnn/AutonomousCar.git"

if not os.path.exists("training/cma_opt.py") and not os.path.isdir("AutonomousCar"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd AutonomousCar
elif not os.path.exists("training/cma_opt.py"):
    %cd AutonomousCar
!git pull -q
# One BLAS thread per rollout worker: prevents thread thrash across workers.
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[var] = "1"
!{sys.executable} -m pip install -q -e ".[train,dev]"
print("cwd:", os.getcwd(), "| cores available:", os.cpu_count())

In [ ]:
# 2) QUICK BENCHMARK — cost of one rollout on this machine (~1-2 s/core is normal)
import time

from training.env import RolloutConfig, run_vector
from training.export import load_policy

_, v, _, _ = load_policy("algorithms/weights/policy_robust_b1.npz")
t0 = time.perf_counter()
r = run_vector(v, RolloutConfig(policy_type="b1", subrays=1, seed=42, max_time_s=20))
dt = time.perf_counter() - t0
print(f"one lap rollout: {dt:.2f} s wall | lap {r.lap_time_s} s | contacts {r.contacts}")
print(f"estimated throughput: ~{os.cpu_count()/dt:.0f} rollouts/s with {os.cpu_count()} workers")

## 3) LAUNCH

Tuned for a many-core Xeon: `--workers 0` auto-detects cores, popsize 32
(≈ 4 + 3·ln(27) minimum, larger = better exploration), **worst-of-6**
generations (each candidate must be clean on the reference track + 5 fresh
random tracks), sensor-noise seed resampled every generation. Warm-started
from the committed robust policy (~9.6 s reference lap).

Adjust `BUDGET_H` and `GENS` to taste; the run also stops by itself after
`STALL` generations without a promotion.


In [ ]:
# 3) LAUNCH — start the training subprocess (self-diagnosing)
import os
import subprocess
import sys  # noqa: F811 - cell must be runnable standalone
import time

GENS = 600          # generation cap
BUDGET_H = 6.0      # wall-clock cap in hours
STALL = 90          # early stop: generations without a promotion
POPSIZE = 32
K = 6               # tracks per generation (reference + K-1 random)

# The simulator loads configs via relative paths: the child must run from
# the repository root, which must also be this notebook's working directory.
if not os.path.exists("training/cma_opt.py"):
    raise RuntimeError(
        "Not in the repository root. Fix with:  %cd <path-to-AutonomousCar>"
    )

cmd = [sys.executable, "-m", "training.cma_opt",
    "--stage", "b1", "--generations", str(GENS), "--popsize", str(POPSIZE),
    "--workers", "0",                      # auto-detect: one worker per core
    "--random-track", "--tracks-per-generation", str(K),
    "--fitness-aggregation", "worst", "--vary-sensor-seed",
    "--reference-track-prob", "0.15",
    "--object-density-range", "0.0", "0.95",
    "--object-scale-range", "1.0", "1.9",
    "--track-width-range", "0.82", "1.0",
    "--init", "algorithms/weights/policy_robust_b1.npz",
    "--sigma0", "0.20", "--seed", "42", "--max-time", "20",
    "--out", "data/training/b1_remote_best.npz",
    "--budget-seconds", str(int(BUDGET_H * 3600)),
    "--stop-after-stall", str(STALL),
]
os.makedirs("data/training", exist_ok=True)
log = open("data/training/remote.log", "w")  # noqa: SIM115 - stays open for the child process
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("started pid", proc.pid, "| workers: auto | popsize", POPSIZE, "| K", K)

# Give the child a moment, then confirm it is actually alive.
time.sleep(5)
if proc.poll() is not None:
    log.close()
    with open("data/training/remote.log") as f:
        print("CHILD EXITED IMMEDIATELY with code", proc.returncode, "\n--- log ---\n" + f.read())
    raise RuntimeError("training subprocess died at startup - see the log above")
print("child is running - use the MONITOR cell to follow progress")


In [ ]:
# 4) MONITOR — run this cell any time (safe to repeat)
import json
import time

try:
    with open("data/training/b1_status.json") as f:
        st = json.load(f)
    print(f"gen {st.get('generation')}/{st.get('generations')} | "
          f"best lap {st.get('best_lap_s')} s | contacts {st.get('best_contacts')} | "
          f"elapsed {time.strftime('%H:%M:%S', time.gmtime(st.get('elapsed_s', 0)))}")
except FileNotFoundError:
    print("no status yet")
print("--- last lines ---")
with open("data/training/remote.log") as f:
    print("".join(f.readlines()[-4:]))


In [ ]:
# 5) STOP — if you want to cut the run short (results so far are kept)
proc.terminate()
proc.wait(timeout=30)
print("stopped")


## 6) VALIDATE

Held-out checks on the winner: reference lap across sensor-noise seeds, then
fresh random tracks (density 0–0.9, object scale 1.0–1.8 — the hard
distribution the robustness objective trains against).


In [ ]:
# 6) VALIDATE — run after (or during) training
BEST = "data/training/b1_objective_best.npz"   # robustness-first winner
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 --max-time 30 --seed-sweep 12
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 3 --max-time 40
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 --max-time 40 --random-sweep 20 --track-seed 999

In [ ]:
# 7) DOWNLOAD — grab these files when the run finishes
for f in ["data/training/b1_objective_best.npz",   # robustness-first winner
          "data/training/b1_remote_best.npz",       # best clean reference lap
          "data/training/b1_log.jsonl",             # full generation history
          "data/training/b1_checkpoint.pkl"]:
    print(f, os.path.exists(f) and os.path.getsize(f), "bytes")
print("\nUpload the .npz files back (GitHub PR or shared drive);")
print("data/training/b1_objective_best.npz is the candidate to ship.")